# L12c: Value Iteration versus Q-Learning
In this lecture, we compare the two ways we have found an optimal policy for a Markov decision process: value iteration, which computes the policy from a model of the world, and Q-learning, which learns it from experience. We look at what each method needs, what each one costs, and where tabular Q-learning runs out of room.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Distinguish model-based from model-free methods:__ A model-based method such as value iteration needs the rewards and the transition probabilities, while a model-free method such as Q-learning needs only a way to try an action and see what happens. We'll write Q-learning as value iteration on the action values with samples in place of the model.
> * __Compare the cost of the two methods:__ Value iteration does a fixed amount of work per sweep that grows with the square of the number of states, while Q-learning does a little work per sample but needs many samples. We'll see how the discount factor drives the work that both methods need.
> * __Recognize the limits of tabular Q-learning:__ A table with one entry per state and action grows too large when the state has many variables, and noisy estimates can make the learned values too optimistic. We'll name these limits and the methods that address them.

Let's get started!
___

## Example
We will use the following example to compare the two methods on the same problem:

> [▶ Adaptive drug dosing with Q-learning](../L12a/CHEME-5800-L12a-Example-AdaptiveDosing-QLearning-Fall-2026.ipynb). In this example, we compute the optimal chemotherapy dosing policy with value iteration from the known dynamics, learn a dosing policy with Q-learning from simulated treatment cycles, and compare the two policies state by state and over a full treatment.

___

## Model-Based and Model-Free Methods
The __model__ of a Markov decision process is its reward function $R(s,a)$ and its transition model $T\left(s^{\prime}\,|\,s,a\right)$. A method that reads the model is __model-based__, and a method that never reads it is __model-free__:
* __Value iteration is model-based.__ Each Bellman backup sums over every next state, weighting each one by its transition probability, so the method needs $R(s,a)$ and $T\left(s^{\prime}\,|\,s,a\right)$ for every state and action.
* __Q-learning is model-free.__ It needs only an environment, a simulator or the real system, that takes an action and reports a reward $r$ and a next state $s^{\prime}$. The environment may use a model inside, but the agent never reads it.

The two methods differ in what they need, what one update does, and what they return. A __sweep__ of value iteration is one pass of the backup over every state:

| | Value iteration | Q-learning |
|---|---|---|
| Needs | the rewards and the transition probabilities | a way to try an action and observe the reward and the next state |
| One update | the backup of one state, a sum over all next states | the update of one state-action pair from one observed move |
| Stops | when a sweep changes no value by more than a tolerance | after a fixed number of episodes |
| Returns | values within $\gamma/(1-\gamma)$ times the tolerance of the optimal values | estimates that approach the optimal values if every pair is tried often enough |

### Q-learning is value iteration with samples
We can run value iteration on the action values instead of the state values. Start from $Q_{0}(s,a)=0$ and apply the Bellman equation for the action values to every state and action in each sweep:

$$
\begin{equation*}
Q_{k+1}(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot\max_{a^{\prime}\in\mathcal{A}}Q_{k}(s^{\prime},a^{\prime})
\end{equation*}
$$

where $k$ counts the sweeps. This __Q-value iteration__ converges to $Q^{*}(s,a)$ for $0\leq\gamma<1$, for the same reason value iteration converges to $U^{*}(s)$: each sweep shrinks the largest error by at least a factor of $\gamma$.

Now compare the right side with the Q-learning target, $r+\gamma\max_{a^{\prime}}Q(s^{\prime},a^{\prime})$. Q-learning makes three changes:
* It replaces the reward $R(s,a)$ with the one reward $r$ it received.
* It replaces the sum over next states with the one next state $s^{\prime}$ it observed.
* It moves $Q(s,a)$ only a fraction $\alpha$ of the way toward the target, so that over many visits the targets average out to the right side of the equation.

Q-learning is Q-value iteration with samples in place of the model, applied to one state-action pair at a time, in the order the agent happens to visit them.

___

## Computational Cost
### Value iteration
Each sweep of value iteration evaluates $|\mathcal{A}|$ actions in each of $|\mathcal{S}|$ states, and each evaluation sums over $|\mathcal{S}|$ next states, so a sweep costs $O(|\mathcal{S}|^{2}\cdot|\mathcal{A}|)$ operations. When each action can reach only a few next states, as in a grid world where each move reaches one square, a sweep that skips the zero transition probabilities costs only $O(|\mathcal{S}|\cdot|\mathcal{A}|)$.

The number of sweeps depends on the discount factor. Each sweep shrinks the largest error in the values by at least a factor of $\gamma$, so after $k$ sweeps the error is at most $\gamma^{k}$ times the starting error. To shrink the error by a factor of 1000, we need $\gamma^{k}\leq 10^{-3}$, or $k\geq\ln(1000)/\ln(1/\gamma)$. That is 66 sweeps for $\gamma=0.9$, but 688 sweeps for $\gamma=0.99$.

For $\gamma$ near one, $\ln(1/\gamma)\approx 1-\gamma$, so the number of sweeps grows like $1/(1-\gamma)$. This quantity has a meaning of its own. The discount weights of all future steps add up to $1+\gamma+\gamma^{2}+\dots = 1/(1-\gamma)$, so $H = 1/(1-\gamma)$ is the __effective horizon__, the number of future steps that a value effectively counts. A common guideline chooses $\gamma\approx 1-1/H$ for a problem whose consequences play out over about $H$ steps: $\gamma\approx 0.9$ for about 10 steps, and $\gamma\approx 0.99$ for about 100 steps. A longer horizon costs more sweeps.

### Q-learning
A Q-learning update costs $O(|\mathcal{A}|)$ operations, the maximum over one row of the table, no matter how many states there are. But each update corrects one entry from one sample, so Q-learning needs many updates:
* __Every pair needs many visits.__ The convergence conditions ask for every state-action pair to be tried again and again, so the number of updates grows with the size of the table.
* __A reward travels back one step per update.__ The target of a state uses the values of the next state, so a reward reaches a state $n$ steps before it only after a chain of $n$ updates, one per state along the way. The longer the effective horizon, the more episodes it takes.
* __Samples have a price.__ On a simulator, a sample costs computer time. On a real system, it costs time, money, or risk: a real robot that learns where the lava is by falling in loses the robot. This is why agents are often trained on a simulator before they act in the world.

Neither method is always cheaper. When a model is available and the state space is small enough to sweep, value iteration is usually faster and more accurate, and it tells us when it has converged. Q-learning is the choice when there is no model, or when the model is too complicated to write down as transition probabilities but easy to simulate one step at a time.

___

## Limitations of Tabular Q-Learning
Tabular Q-learning stores one number for every state-action pair. That table is both its strength, since every entry converges to its own optimal value, and its main limitation:
* __Size of the table:__ A state with $m$ variables, each rounded to $n$ levels, has $n^{m}$ states. Two variables on 41 levels give 1681 states, but ten variables on 41 levels give about $1.3\times 10^{16}$ states, more than any table can hold and far more than an agent could visit.
* __Discretization:__ A continuous state, such as a tumor size or a concentration, must be rounded to grid levels. A coarse grid treats different states as the same state, and a fine grid makes the table larger.
* __Delayed rewards:__ A reward that arrives only at the end of a long episode reaches the early states one update at a time, so the agent needs many episodes before the early decisions improve. Reward shaping can help, by adding small rewards along the way.

* __Overestimation:__ The target takes the maximum of the estimates in the next state, and the maximum of noisy estimates can be too large on average, even when each estimate is right on average. For example, if every action of a state has a true value of zero, but each estimate is off by random noise, the largest estimate is above zero on average, and the update passes this optimism back to earlier states. [Double Q-learning (van Hasselt, 2010)](https://papers.nips.cc/paper_files/paper/2010/hash/091d584fced301b442654dd8c23b3fc9-Abstract.html) keeps two tables, updates one of them at random after each move, and uses one table to choose the best next action and the other to evaluate it, which reduces the upward bias of the maximum.

The usual answer to the size of the table is to replace it with a function that computes $Q(s,a)$ from __features__ of the state, numbers that describe it such as the position of a robot or the temperature of a reactor, so that what the agent learns in one state carries over to similar states. [Deep Q-learning (Mnih et al., 2015)](https://www.nature.com/articles/nature14236) uses a __neural network__, a flexible function with many adjustable parameters fitted to data, for this function and learned to play Atari video games from the pixels on the screen.

___

## Lab
In the [lab](../L12d/CHEME-5800-L12d-Lab-ExplorationAndReproducibility-Fall-2026.ipynb), we learn a feeding policy for a bioreactor with Q-learning, see how the random seed and the amount of exploration change what the agent learns, and use the visit counts to diagnose a policy that fails.

___

## Summary
In this lecture, we compared value iteration, which computes the optimal policy from a model, with Q-learning, which learns it from experience.

> __Key Takeaways:__
>
> * __Q-learning is value iteration with samples:__ Value iteration on the action values sums over every next state, weighting each by its transition probability. Q-learning replaces that sum with the one next state and reward it observes, and moves its estimate only part of the way toward them.
>
> * __The discount factor sets the work for both methods:__ As the discount factor approaches one, value iteration needs more sweeps, and a reward has to travel back over more steps before Q-learning can use it. Value iteration pays for the size of the model in every sweep, while Q-learning pays for the many samples it needs.
>
> * __The table limits tabular Q-learning:__ One entry per state and action grows exponentially with the number of state variables, and continuous states must be rounded onto a grid. Methods that compute the action values from features of the state, such as deep Q-learning, address the size of the table.

When we have a model of a small world, value iteration is the faster and surer choice; when we have only experience, Q-learning can still find the optimal policy, at the price of many samples.

___